In [1]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

os.chdir("/Users/ethan.bcht/Documents/cours/M2/Master thesis/code")
print("Imports OK — matplotlib non chargé")

Imports OK — matplotlib non chargé


In [2]:
panel = pd.read_parquet("data/panel_monthly.parquet")
events = pd.read_csv("data/events.csv")
events["date"] = pd.to_datetime(events["date"])
panel = panel[panel["R2_raw"] > 0].copy()
panel["date"] = pd.to_datetime(panel["date"])

print(f"Panel : {len(panel):,} lignes | {panel['ticker'].nunique()} tickers")
print(f"Fenêtre : {panel['date'].min().date()} → {panel['date'].max().date()}")
print(f"Colonnes : {panel.columns.tolist()}")
panel.head(3)

Panel : 130,630 lignes | 981 tickers
Fenêtre : 2013-01-01 → 2026-02-01
Colonnes : ['date', 'ticker', 'R2_raw', 'Synchronicity', 'Idio_Vol', 'Amihud', 'Turnover', 'N_obs']


,date,ticker,R2_raw,Synchronicity,Idio_Vol,Amihud,Turnover,N_obs
0,2013-01-01,11B.WA,0.071132,-2.569432,0.019208,4.655719,903.60,20
1,2013-02-01,11B.WA,0.060235,-2.747373,0.036383,2.243370,2289.30,20
2,2013-03-01,11B.WA,0.228217,-1.218407,0.024544,4.743194,531.75,20


In [4]:
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        "Mean Synchronicity over time",
        "Mean R² (raw) over time",
        "Mean Idiosyncratic Volatility over time",
        "ADD events per rebalancing date",
        "Distribution of R²",
        "Synchronicity by country",
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

# ── 1. Mean Synchronicity ──────────────────────────────────
monthly_synch = panel.groupby("date")["Synchronicity"].mean()
fig.add_trace(go.Scatter(x=monthly_synch.index, y=monthly_synch.values,
    mode="lines", name="Synchronicity", line=dict(color="#1f77b4", width=1.5)), row=1, col=1)
fig.add_hline(y=monthly_synch.mean(), line_dash="dash", line_color="red",
    annotation_text=f"Mean={monthly_synch.mean():.2f}", row=1, col=1)

# ── 2. Mean R² ────────────────────────────────────────────
monthly_r2 = panel.groupby("date")["R2_raw"].mean()
fig.add_trace(go.Scatter(x=monthly_r2.index, y=monthly_r2.values,
    mode="lines", name="R²", line=dict(color="#ff7f0e", width=1.5)), row=1, col=2)
fig.add_hline(y=monthly_r2.mean(), line_dash="dash", line_color="red",
    annotation_text=f"Mean={monthly_r2.mean():.3f}", row=1, col=2)

# ── 3. Mean Idio Vol ──────────────────────────────────────
monthly_vol = panel.groupby("date")["Idio_Vol"].mean()
fig.add_trace(go.Scatter(x=monthly_vol.index, y=monthly_vol.values,
    mode="lines", name="Idio Vol", line=dict(color="#2ca02c", width=1.5)), row=2, col=1)
fig.add_hline(y=monthly_vol.mean(), line_dash="dash", line_color="red",
    annotation_text=f"Mean={monthly_vol.mean():.4f}", row=2, col=1)

# ── 4. ADD events bar ─────────────────────────────────────
add_counts = events[events["event_type"] == "ADD"].groupby("date").size().reset_index(name="n")
fig.add_trace(go.Bar(x=add_counts["date"], y=add_counts["n"],
    name="ADD events", marker_color="#9467bd", opacity=0.8), row=2, col=2)

# ── 5. Distribution R² ───────────────────────────────────
fig.add_trace(go.Histogram(x=panel["R2_raw"], nbinsx=80,
    name="R² dist", marker_color="#8c564b", showlegend=False), row=3, col=1)
fig.add_vline(x=panel["R2_raw"].median(), line_dash="dash", line_color="red",
    annotation_text=f"Median={panel['R2_raw'].median():.3f}", row=3, col=1)

# ── 6. Synchronicity by country ───────────────────────────
panel_comp = pd.read_parquet("data/panel_composition.parquet")[["ticker", "Country"]].drop_duplicates("ticker")
panel_c = panel.merge(panel_comp, on="ticker", how="left")
top_countries = ["GB", "FR", "DE", "SE", "CH", "IT", "NO", "ES", "NL", "DK"]
for country in top_countries:
    vals = panel_c[panel_c["Country"] == country]["Synchronicity"].dropna()
    fig.add_trace(go.Box(y=vals, name=country, marker_color="#aec7e8",
        showlegend=False, boxpoints=False), row=3, col=2)

fig.update_layout(
    height=1100,
    title_text="STOXX Europe 600 — Market Efficiency Panel (2013–2026)",
    title_font_size=15,
    showlegend=False,
)

os.makedirs("figures", exist_ok=True)
fig.write_image("figures/panel_overview.png", width=1400, height=1100, scale=2)
print("Saved: figures/panel_overview.png")
fig.show()


ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
